# Stress Testing: Portfolio-Level Credit Risk Under Adverse Scenarios

## Objective

This notebook evaluates how a rule-based credit risk scoring framework behaves under adverse borrower-level stress scenarios.

Unlike the baseline notebook, which constructs the initial pseudo probability of default (pseudo PD), and unlike sensitivity analysis, which applies relatively small isolated changes, this notebook focuses on **systematic deterioration** in borrower conditions.

The goal is to simulate how overall portfolio risk changes when multiple borrower characteristics worsen at the same time.

---

## Why Stress Testing Matters

Stress testing is important in credit risk because it helps answer the following question:

> How would the risk profile of a borrower portfolio change if financial conditions deteriorate materially?

This is useful for:

- understanding portfolio vulnerability
- identifying risk migration under adverse conditions
- evaluating whether the scoring framework responds in a directionally sensible way
- demonstrating scenario-based model validation logic

---

## Stress Scenario Design

This notebook constructs two adverse scenarios:

### Mild Stress Scenario
- Credit amount increases by 20%
- Duration increases by 30%
- Saving accounts deteriorate by one category
- Checking account deteriorates by one category

### Severe Stress Scenario
- Credit amount increases by 30%
- Duration increases by 50%
- Saving accounts are set to "little"
- Checking account is set to "little"

These shocks are designed to represent worsening borrower burden and weaker liquidity conditions.

---

## Method

The notebook follows these steps:

1. Load the raw credit dataset
2. Build the baseline rule-based risk score
3. Convert the risk score into a pseudo probability of default
4. Independently reload the raw data for each stress scenario
5. Apply stress shocks
6. Recompute pseudo PD under each scenario
7. Compare:
   - average pseudo PD
   - risk bucket distributions
   - borrower-level PD changes

---

## Important Note

This is not a production stress testing system and does not use macroeconomic forecasting or statistically estimated default models.

Instead, it is a simplified demonstration of how an interpretable credit risk framework can be used for scenario analysis and portfolio stress testing.

The emphasis is on:

- interpretability
- consistency
- risk direction
- scenario usefulness

In [10]:
import pandas as pd
import numpy as np

# =====================================
# 1. Define file path
# =====================================
file_path = "/Users/quentingao/Desktop/german_credit_data.csv"

# =====================================
# 2. Define scoring function
# This function is fully self-contained
# =====================================
def build_credit_risk_framework(df_raw):
    df = df_raw.copy()

    # -----------------------------
    # Basic cleaning
    # -----------------------------
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()

    df = df.replace({"nan": np.nan, "NA": np.nan})

    # -----------------------------
    # Map categorical variables into scores
    # Higher score = safer
    # -----------------------------
    saving_map = {
        "little": 0,
        "moderate": 1,
        "quite rich": 2,
        "rich": 3
    }

    checking_map = {
        "little": 0,
        "moderate": 1,
        "rich": 2
    }

    housing_map = {
        "rent": 0,
        "free": 1,
        "own": 2
    }

    df["saving_score"] = df["Saving accounts"].map(saving_map)
    df["saving_score"] = df["saving_score"].fillna(0.5)

    df["checking_score"] = df["Checking account"].map(checking_map)
    df["checking_score"] = df["checking_score"].fillna(0.5)

    df["housing_score"] = df["Housing"].map(housing_map)
    df["job_score"] = df["Job"]

    # -----------------------------
    # Standardize continuous variables
    # -----------------------------
    continuous_vars = ["Age", "Credit amount", "Duration"]

    for col in continuous_vars:
        mean_col = df[col].mean()
        std_col = df[col].std()
        df[f"{col}_z"] = (df[col] - mean_col) / std_col

    # -----------------------------
    # Build rule-based risk score
    # Higher score = riskier
    # -----------------------------
    df["risk_score"] = (
        0.60 * df["Credit amount_z"]
        + 0.75 * df["Duration_z"]
        - 0.25 * df["Age_z"]
        - 0.80 * df["saving_score"]
        - 0.60 * df["checking_score"]
        - 0.30 * df["housing_score"]
        - 0.20 * df["job_score"]
    )

    # -----------------------------
    # Convert to pseudo PD
    # -----------------------------
    df["pseudo_pd"] = 1 / (1 + np.exp(-df["risk_score"]))

    # -----------------------------
    # Create risk buckets
    # -----------------------------
    df["risk_bucket"] = pd.cut(
        df["pseudo_pd"],
        bins=[0, 0.20, 0.40, 0.60, 1.00],
        labels=["Low", "Moderate", "Elevated", "High"],
        include_lowest=True
    )

    return df

# =====================================
# 3. Load raw data and build baseline
# =====================================
df_base_raw = pd.read_csv(file_path)
df_base = build_credit_risk_framework(df_base_raw)

print("Baseline pseudo PD summary:")
print(df_base["pseudo_pd"].describe())

print("\nBaseline risk bucket distribution:")
print(df_base["risk_bucket"].value_counts().sort_index())

# =====================================
# 4. Mild Stress Scenario
# Reload raw data independently
# =====================================
df_mild_raw = pd.read_csv(file_path)

if "Unnamed: 0" in df_mild_raw.columns:
    df_mild_raw = df_mild_raw.drop(columns=["Unnamed: 0"])

# Apply mild stress shocks
df_mild_raw["Credit amount"] = df_mild_raw["Credit amount"] * 1.20
df_mild_raw["Duration"] = df_mild_raw["Duration"] * 1.30

# Deteriorate saving accounts by one category
df_mild_raw["Saving accounts"] = df_mild_raw["Saving accounts"].replace({
    "rich": "quite rich",
    "quite rich": "moderate",
    "moderate": "little"
})

# Deteriorate checking account by one category
df_mild_raw["Checking account"] = df_mild_raw["Checking account"].replace({
    "rich": "moderate",
    "moderate": "little"
})

df_mild = build_credit_risk_framework(df_mild_raw)

print("\nMild stress pseudo PD summary:")
print(df_mild["pseudo_pd"].describe())

print("\nMild stress risk bucket distribution:")
print(df_mild["risk_bucket"].value_counts().sort_index())

# =====================================
# 5. Severe Stress Scenario
# Reload raw data independently
# =====================================
df_severe_raw = pd.read_csv(file_path)

if "Unnamed: 0" in df_severe_raw.columns:
    df_severe_raw = df_severe_raw.drop(columns=["Unnamed: 0"])

# Apply severe stress shocks
df_severe_raw["Credit amount"] = df_severe_raw["Credit amount"] * 1.30
df_severe_raw["Duration"] = df_severe_raw["Duration"] * 1.50

df_severe_raw["Saving accounts"] = "little"
df_severe_raw["Checking account"] = "little"

df_severe = build_credit_risk_framework(df_severe_raw)

print("\nSevere stress pseudo PD summary:")
print(df_severe["pseudo_pd"].describe())

print("\nSevere stress risk bucket distribution:")
print(df_severe["risk_bucket"].value_counts().sort_index())

# =====================================
# 6. Portfolio-level comparison
# =====================================
portfolio_summary = pd.DataFrame({
    "Scenario": ["Baseline", "Mild Stress", "Severe Stress"],
    "Average_Pseudo_PD": [
        df_base["pseudo_pd"].mean(),
        df_mild["pseudo_pd"].mean(),
        df_severe["pseudo_pd"].mean()
    ],
    "Median_Pseudo_PD": [
        df_base["pseudo_pd"].median(),
        df_mild["pseudo_pd"].median(),
        df_severe["pseudo_pd"].median()
    ],
    "High_Risk_Share_PD_gt_0_5": [
        (df_base["pseudo_pd"] > 0.5).mean(),
        (df_mild["pseudo_pd"] > 0.5).mean(),
        (df_severe["pseudo_pd"] > 0.5).mean()
    ]
})

print("\nPortfolio-level comparison:")
print(portfolio_summary)

# =====================================
# 7. Borrower-level PD comparison
# =====================================
borrower_compare = pd.DataFrame({
    "baseline_pd": df_base["pseudo_pd"],
    "mild_stress_pd": df_mild["pseudo_pd"],
    "severe_stress_pd": df_severe["pseudo_pd"]
})

borrower_compare["delta_mild"] = borrower_compare["mild_stress_pd"] - borrower_compare["baseline_pd"]
borrower_compare["delta_severe"] = borrower_compare["severe_stress_pd"] - borrower_compare["baseline_pd"]

print("\nBorrower-level change under mild stress:")
print(borrower_compare["delta_mild"].describe())

print("\nBorrower-level change under severe stress:")
print(borrower_compare["delta_severe"].describe())

print("\nShare of borrowers with higher PD under mild stress:")
print((borrower_compare["delta_mild"] > 0).mean())

print("\nShare of borrowers with higher PD under severe stress:")
print((borrower_compare["delta_severe"] > 0).mean())

# =====================================
# 8. Display top borrowers with largest increase
# =====================================
top_mild = df_base.copy()
top_mild["baseline_pd"] = df_base["pseudo_pd"]
top_mild["mild_stress_pd"] = df_mild["pseudo_pd"]
top_mild["delta_mild"] = borrower_compare["delta_mild"]

print("\nTop 10 borrowers with largest PD increase under mild stress:")
print(
    top_mild[
        [
            "Age", "Sex", "Job", "Housing", "Saving accounts", "Checking account",
            "Credit amount", "Duration", "Purpose",
            "baseline_pd", "mild_stress_pd", "delta_mild"
        ]
    ]
    .sort_values("delta_mild", ascending=False)
    .head(10)
)

top_severe = df_base.copy()
top_severe["baseline_pd"] = df_base["pseudo_pd"]
top_severe["severe_stress_pd"] = df_severe["pseudo_pd"]
top_severe["delta_severe"] = borrower_compare["delta_severe"]

print("\nTop 10 borrowers with largest PD increase under severe stress:")
print(
    top_severe[
        [
            "Age", "Sex", "Job", "Housing", "Saving accounts", "Checking account",
            "Credit amount", "Duration", "Purpose",
            "baseline_pd", "severe_stress_pd", "delta_severe"
        ]
    ]
    .sort_values("delta_severe", ascending=False)
    .head(10)
)

Baseline pseudo PD summary:
count    1000.000000
mean        0.237174
std         0.225776
min         0.004137
25%         0.077399
50%         0.154343
75%         0.327882
max         0.983426
Name: pseudo_pd, dtype: float64

Baseline risk bucket distribution:
risk_bucket
Low         600
Moderate    205
Elevated     96
High         99
Name: count, dtype: int64

Mild stress pseudo PD summary:
count    1000.000000
mean        0.279556
std         0.235833
min         0.012261
25%         0.109450
50%         0.194661
75%         0.378742
max         0.983426
Name: pseudo_pd, dtype: float64

Mild stress risk bucket distribution:
risk_bucket
Low         511
Moderate    255
Elevated    104
High        130
Name: count, dtype: int64

Severe stress pseudo PD summary:
count    1000.000000
mean        0.327315
std         0.233984
min         0.032381
25%         0.151296
50%         0.244249
75%         0.428125
max         0.987668
Name: pseudo_pd, dtype: float64

Severe stress risk bucket 

# Stress Testing Results Summary

## Main Finding

Both adverse scenarios increase the average pseudo probability of default relative to the baseline case.

The increase is larger under the severe stress scenario, indicating that the portfolio becomes meaningfully riskier when borrower burden rises and liquidity weakens substantially.

---

## Portfolio-Level Effect

The stress scenarios shift the overall credit risk distribution upward.

In particular:

- average pseudo PD rises
- median pseudo PD rises
- the share of borrowers classified as high risk increases

This suggests that the portfolio is sensitive to broad-based deterioration in borrower conditions.

---

## Borrower-Level Effect

At the individual borrower level, most borrowers experience an increase in pseudo PD under both stress scenarios.

The increase is largest for borrowers who are already relatively vulnerable under the baseline framework, especially those with:

- weaker savings positions
- weaker checking balances
- larger credit amounts
- longer durations

---

## Interpretation

The stress test results are directionally consistent with standard credit risk intuition:

- larger debt burden increases risk
- weaker liquidity increases risk
- longer repayment horizon increases uncertainty

This supports the internal consistency of the rule-based scoring framework.

---

## Conclusion

This notebook demonstrates how an interpretable credit risk scoring framework can be extended from baseline scoring into portfolio-level stress testing.

Although simplified, the framework provides a useful demonstration of:

- scenario analysis
- risk migration
- borrower vulnerability under adverse conditions